**Evaluating Deep Neural Network Performance on an Imbalanced Stroke Prediction Dataset**


In [ ]:
# import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import tensorflow as tf
from tensorflow.keras import layers as L, callbacks as C

import warnings
warnings.filterwarnings('ignore')

#### **Load data and preprocess**
- This is the stroke dataset which contains 5110 observations with 12 attributes.
- Kaggle link: [Stroke Prediction Dataset](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset)

In [2]:
# load dataset
heart_stroke_df = pd.read_csv('../data/healthcare-dataset-stroke-data.csv')
print("Shape of the dataset:", heart_stroke_df.shape)
heart_stroke_df.head()

Shape of the dataset: (5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [3]:
heart_stroke_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [4]:
# drop rows with missing values
heart_stroke_df = heart_stroke_df.dropna()
print("Shape of the dataset after dropping missing values:", heart_stroke_df.shape)
heart_stroke_df.head()

Shape of the dataset after dropping missing values: (4909, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
5,56669,Male,81.0,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1


In [5]:
# encode categorical variables into numerical format
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
le = LabelEncoder()
for col in categorical_cols:
    heart_stroke_df[col] = le.fit_transform(heart_stroke_df[col])

In [6]:
# define features X and target y 
X = heart_stroke_df.drop(columns=['stroke'])
y = heart_stroke_df['stroke']

In [7]:
y.value_counts()

stroke
0    4700
1     209
Name: count, dtype: int64

In [ ]:
# Split the data into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Standardize the features
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

# Convert to float32 and int64 for TensorFlow compatibility
Xtr = X_train_s.astype("float32")
ytr = y_train.values.astype("int64")
Xva = X_valid_s.astype("float32")
yva = y_valid.values.astype("int64")
Xte = X_test_s.astype("float32")
yte = y_test.values.astype("int64")

### **Implementation 1**
Baseline model with 
- Learning rate = 1e-3
- Epochs = 100
- Batch size = 64
- Only two dense layers (128 + 64)

In [ ]:
# define the MLP model with batch normalization and dropout
def build_mlp(input_dim, num_classes=2, p_drop=0.2):
    inputs = tf.keras.Input(shape=(input_dim,))
    
    x = L.Dense(128, kernel_initializer="he_normal", use_bias=True)(inputs)
    x = L.BatchNormalization()(x)
    x = L.ReLU()(x)
    x = L.Dropout(p_drop)(x)

    x = L.Dense(64, kernel_initializer="he_normal", use_bias=True)(x)
    x = L.BatchNormalization()(x)
    x = L.ReLU()(x)
    x = L.Dropout(p_drop)(x)

    outputs = L.Dense(num_classes, kernel_initializer="he_normal", use_bias=True)(x)
    return tf.keras.Model(inputs, outputs, name="MLP_BN_Dropout")

In [ ]:
# build and summarize the model
model = build_mlp(input_dim=Xtr.shape[1], num_classes=2, p_drop=0.2)
model.summary()

Model: "MLP_BN_Dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 11)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,690 (41.76 KB)

 Trainable params: 10,306 (40.26 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
# compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),  # Original LR
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

# callbacks for the model
es = C.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
rlr = C.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1)

In [ ]:
# train the model
history = model.fit(
    Xtr, ytr,
    validation_data=(Xva, yva),
    epochs=100,         # Original epochs
    batch_size=64,      # Original batch size
    callbacks=[es, rlr],
    verbose=1
)

Epoch 1/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8586 - loss: 0.3581 - val_accuracy: 0.9497 - val_loss: 0.2146 - learning_rate: 0.0010
Epoch 2/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9497 - loss: 0.1892 - val_accuracy: 0.9552 - val_loss: 0.1758 - learning_rate: 0.0010
Epoch 3/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9558 - loss: 0.1600 - val_accuracy: 0.9565 - val_loss: 0.1649 - learning_rate: 0.0010
Epoch 4/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9526 - loss: 0.1604 - val_accuracy: 0.9579 - val_loss: 0.1553 - learning_rate: 0.0010
Epoch 5/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9563 - loss: 0.1496 - val_accuracy: 0.9579 - val_loss: 0.1561 - learning_rate: 0.0010
Epoch 6/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9552 - loss: 0.1487 - val_accuracy: 0.9579 - val_loss: 0.1559 - learning_rate: 0.0010
Epoch 7/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9552 - loss: 0.1442 - val_acc

In [ ]:
# evaluate the model
test_loss, test_acc = model.evaluate(Xte, yte, verbose=0)
print(f"\nTest loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")


Test loss: 0.1674 | Test accuracy: 0.9566


In [ ]:
# make predictions
logits = model.predict(Xte, batch_size=256, verbose=0)
probs = tf.nn.softmax(logits, axis=1).numpy()
p1 = probs[:, 1]
y_pred = (p1 >= 0.5).astype(int)

# report results
print("\nROC-AUC (class 1):", roc_auc_score(yte, p1))
print("\nConfusion matrix:\n", confusion_matrix(yte, y_pred))
print("\nClassification report:\n", classification_report(yte, y_pred, digits=4))


ROC-AUC (class 1): 0.7646719858156028

Confusion matrix:
 [[705   0]
 [ 32   0]]

Classification report:
               precision    recall  f1-score   support

           0     0.9566    1.0000    0.9778       705
           1     0.0000    0.0000    0.0000        32

    accuracy                         0.9566       737
   macro avg     0.4783    0.5000    0.4889       737
weighted avg     0.9150    0.9566    0.9354       737



### **Implementation 2**
Baseline model with 
- Learning rate = 5e-4
- Epochs = 150
- Batch size = 32 
- 3 dense layers

In [ ]:
# define the model with an additional dense layer
def build_mlp_updated(input_dim, num_classes=2, p_drop=0.2):
    inputs = tf.keras.Input(shape=(input_dim,))
    
    # Layer 1
    x = L.Dense(128, kernel_initializer="he_normal", use_bias=True)(inputs)
    x = L.BatchNormalization()(x)
    x = L.ReLU()(x)
    x = L.Dropout(p_drop)(x)
    
    # Layer 2
    x = L.Dense(64, kernel_initializer="he_normal", use_bias=True)(x)
    x = L.BatchNormalization()(x)
    x = L.ReLU()(x)
    x = L.Dropout(p_drop)(x)
    
    # Layer 3
    x = L.Dense(32, kernel_initializer="he_normal", use_bias=True)(x)
    x = L.BatchNormalization()(x)
    x = L.ReLU()(x)
    x = L.Dropout(p_drop)(x)
    
    # Output layer (logits)
    outputs = L.Dense(num_classes, kernel_initializer="he_normal", use_bias=True)(x)
    
    return tf.keras.Model(inputs, outputs, name="MLP_BN_Dropout_ExtraLayer")

In [20]:
# build updated model
model1 = build_mlp_updated(input_dim=Xtr.shape[1], num_classes=2, p_drop=0.2)
model1.summary()

Model: "MLP_BN_Dropout_ExtraLayer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 11)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,834 (50.13 KB)

 Trainable params: 12,386 (48.38 KB)

 Non-trainable params: 448 (1.75 KB)

In [21]:
# complile updated model with a lower learning rate
model1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),  # lowered LR
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [22]:
# callbacks for updated model
es = C.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
rlr = C.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1)


In [23]:
# train updated model with more epochs and smaller batch size
history = model1.fit(
    Xtr, ytr,
    validation_data=(Xva, yva),
    epochs=150,         # increased epochs
    batch_size=32,      # smaller batch
    callbacks=[es, rlr],
    verbose=1
)

Epoch 1/150
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7395 - loss: 0.5878 - val_accuracy: 0.9226 - val_loss: 0.3858 - learning_rate: 5.0000e-04
Epoch 2/150
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9255 - loss: 0.2824 - val_accuracy: 0.9538 - val_loss: 0.2362 - learning_rate: 5.0000e-04
Epoch 3/150
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9479 - loss: 0.2054 - val_accuracy: 0.9565 - val_loss: 0.1953 - learning_rate: 5.0000e-04
Epoch 4/150
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9558 - loss: 0.1839 - val_accuracy: 0.9565 - val_loss: 0.1777 - learning_rate: 5.0000e-04
Epoch 5/150
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9511 - loss: 0.1775 - val_accuracy: 0.9579 - val_loss: 0.1729 - learning_rate: 5.0000e-04
Epoch 6/150
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9534 - loss: 0.1700 - val_accuracy: 0.9565 - val_loss: 0.1686 - learning_rate: 5.0000e-04
Epoch 7/150
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - acc

In [24]:
# evaluate updated model
test_loss, test_acc = model1.evaluate(Xte, yte, verbose=0)
print(f"\nTest loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")


Test loss: 0.1614 | Test accuracy: 0.9566


In [25]:
# predict with updated model
logits = model1.predict(Xte, batch_size=256, verbose=0)
probs = tf.nn.softmax(logits, axis=1).numpy()
p1 = probs[:, 1]
y_pred = (p1 >= 0.5).astype(int)

# ======== REPORTS ========
print("\nROC-AUC (class 1):", roc_auc_score(yte, p1))
print("\nConfusion matrix:\n", confusion_matrix(yte, y_pred))
print("\nClassification report:\n", classification_report(yte, y_pred, digits=4))


ROC-AUC (class 1): 0.7737145390070923

Confusion matrix:
 [[705   0]
 [ 32   0]]

Classification report:
               precision    recall  f1-score   support

           0     0.9566    1.0000    0.9778       705
           1     0.0000    0.0000    0.0000        32

    accuracy                         0.9566       737
   macro avg     0.4783    0.5000    0.4889       737
weighted avg     0.9150    0.9566    0.9354       737



#### **Interpretation**

- Overall, the accuracy looks excellent with 95.66%, but its entirely due to predicitng the majority class (no stroke)
- The confusion matrix shows every single sample was predicted as class 0. 
- The recall for class 1 is 0 which means the model never recognizes any stroke patients.


Adjusting the parameters
- After reducing the learning rate for implementation 2, the convergence was smoother which possibly mean better generalization. It also has slightly lower test loss (0.1674 -> 0.1614).
- When number of epochs are increased from 100 to 150, nothing changed other than the training time.
- When batch size is rediuced from 64 to 32, there is a little improvement in loss/AUC; and it trains slow but a bit more stable.
- When the dense layers increased from 2 (128 + 64) to 3 (128 + 64 + 32), the loss was slightly lower but no improvement in recall or accuracy for class 1.
